- ID: S
- 本編: [2.04.多次元配列](https://atcoder.jp/contests/APG4bPython/tasks/APG4bPython_s)
- 練習問題: [EX17.ゲーム大会](https://atcoder.jp/contests/APG4bPython/tasks/APG4bPython_cf)

## 二次元配列を作成する / 二次元配列を作成する際の注意 / 二次元配列の初期化方法の注意

- `[0] * 2`
- `[[0] * 2] * 3`

In [13]:
# 初期化
x = 0
y = [x] * 2
z = [y] * 3
print(x, y, z)

0 [0, 0] [[0, 0], [0, 0], [0, 0]]


In [14]:
# 変更
# x = 1
# y[0] = 1
z[0][0] = 1
print(x, y, z)

# -> y, z[0], z[1], z[2]

0 [1, 0] [[1, 0], [1, 0], [1, 0]]


## なぜこうなるのか

Python では、変数は「値そのもの」ではなく「オブジェクトへの参照（アドレス）」を持つ。

```
x = 0       x ──→ [オブジェクト 0]

y = [x] * 2
            y ──→ [リスト: 0, 0]

z = [y] * 3
            z ──→ [参照, 参照, 参照]
                       │      │      │
                       └──────┴──────┘
                              ↓
                         y と同じリスト
```

`[y] * 3` は「y のコピーを3つ作る」のではなく、**同じリストオブジェクト y への参照を3つ並べたリスト**を作る。

そのため `z[0]`, `z[1]`, `z[2]` はすべて **同一のリストオブジェクト** を指している。

```python
z[0][0] = 1
# これは実質的に y[0] = 1 と同じ
# z[0], z[1], z[2] がすべて y を指しているので全行が変わる
```

`id()` 関数でオブジェクトのアドレスを確認できる。

In [15]:
# id() でアドレスを確認
y = [0] * 2
z = [y] * 3

print(id(z[0]))  # z[0], z[1], z[2] はすべて同じアドレス
print(id(z[1]))
print(id(z[2]))
print(id(y))     # y も同じアドレス

4463250432
4463250432
4463250432
4463250432


## なぜ `[0] * 2` はOKで `[[0, 0]] * 3` はNGなのか

どちらも「同じオブジェクトへの参照を並べる」という点は同じ。違いは要素の**ミュータビリティ（変更可能かどうか）**にある。

### `[0] * 2` がOKな理由

整数 `0` は**イミュータブル（変更不可）**なオブジェクト。

```
y = [0] * 2
y ──→ [参照, 参照]
         │      │
         └──────┘
              ↓
         オブジェクト 0（変更できない）
```

`y[0] = 1` とすると、**オブジェクト `0` を `1` に書き換えるのではなく**、  
インデックス 0 が指す先を「別のオブジェクト `1`」に付け替えるだけ。

```
y[0] = 1 の後:
y ──→ [参照, 参照]
         │      │
         ↓      ↓
オブジェクト 1  オブジェクト 0  ← それぞれ独立
```

他の要素は影響を受けない。

---

### `[[0, 0]] * 3` がNGな理由

リストは**ミュータブル（変更可能）**なオブジェクト。

```
z = [[0, 0]] * 3
z ──→ [参照, 参照, 参照]
          │      │      │
          └──────┴──────┘
                 ↓
          リスト [0, 0]（変更できてしまう）
```

`z[0][0] = 1` とすると、参照先の**リストオブジェクト自体を書き換える**。  
`z[0]`, `z[1]`, `z[2]` は全員同じリストを指しているので、全行が変わる。

---

### まとめ

| | `[0] * 2` | `[[0,0]] * 3` |
|---|---|---|
| 要素の型 | `int`（イミュータブル） | `list`（ミュータブル） |
| `=` で要素を変えると | 参照先を付け替えるだけ → 他に影響なし | オブジェクト自体を書き換える → 全参照に影響 |
| 結果 | OK | NG |

## なぜミュータブル・イミュータブルという概念が必要なのか

### 前提：Pythonの変数はすべてポインタ

C言語などでは「値をそのまま持つ変数」と「ポインタ（アドレスを持つ変数）」を明示的に使い分ける。  
Pythonでは**すべての変数がポインタ**（参照）であり、値を直接持つ変数は存在しない。

```
# C言語
int x = 5;      // x はメモリ上の箱に 5 という値を直接持つ
int *p = &x;    // p は x のアドレスを持つポインタ

# Python
x = 5           # x は「5 というオブジェクト」へのポインタ
y = x           # y も同じオブジェクトを指す（コピーではない）
```

```
メモリのイメージ:

  変数名        オブジェクト
  x ─────────→ [int: 5]
  y ─────────↗
```

全員がポインタである以上、**「参照を共有したとき、相手に値を書き換えられるリスク」が常に存在する**。  
これを制御するための仕組みがミュータビリティ。

---

### ミュータビリティが必要な理由

#### 1. 参照の共有を安全にするため

```python
def double(lst):
    for i in range(len(lst)):
        lst[i] *= 2   # 元のリストを直接変更してしまう

data = [1, 2, 3]
double(data)
print(data)   # [2, 4, 6] ← 呼び出し元が知らないうちに変わっている
```

関数にオブジェクトを渡すとき、Pythonは**参照をそのまま渡す**（コピーしない）。  
ミュータブルなオブジェクトは関数の中で書き換えられてしまう可能性がある。  
イミュータブルなオブジェクトは構造的に書き換えが不可能なので、この問題が起きない。

```
# 参照渡しのイメージ
data ──→ [リスト: 1, 2, 3]
              ↑
         lst も同じリストを指す（コピーではない）
         → lst を変更すると data も変わる
```

#### 2. 辞書のキーやセットに使えるかどうか

辞書（`dict`）のキーやセット（`set`）は、内部でハッシュ値を使って管理している。  
オブジェクトが途中で変わると、ハッシュ値も変わってしまい、もう見つけられなくなる。

```python
d = {}
key = [1, 2]   # list はミュータブル
d[key] = "a"   # TypeError: unhashable type: 'list'

key = (1, 2)   # tuple はイミュータブル
d[key] = "a"   # OK
```

**イミュータブルである = ハッシュ値が変わらない = 辞書のキーに使える**

#### 3. メモリ効率のための参照共有を安全にするため

前述の通り、Pythonは `0`〜`256` の整数をキャッシュして使い回す。  
これが安全にできるのは `int` がイミュータブルだから。

```
a = 1
b = 1
# a と b は同じオブジェクトを指す（メモリ節約）
# int は変更できないので、a を通じて b の値が変わる心配がない
```

もし `int` がミュータブルだったら、`a` を書き換えると `b` も変わってしまい大混乱になる。

#### 4. マルチスレッドの安全性

複数のスレッドが同じオブジェクトを同時に読み書きすると、データが壊れる（競合状態）。  
イミュータブルなオブジェクトは書き換えが起きないので、複数スレッドで安全に共有できる。

---

### まとめ：C言語との比較

| | C言語 | Python |
|---|---|---|
| 変数の実体 | 値 or ポインタを明示的に選ぶ | 常にポインタ（参照） |
| 書き換え制御 | `const` でポインタ/値を保護 | イミュータブルな型で制御 |
| 参照渡し | `&` で明示的に渡す | 常に参照渡し（自動） |
| 変更不可の保証 | `const int *p` など | `int`, `str`, `tuple` などイミュータブルな型を使う |

Pythonでは「すべてがポインタ」という設計の代償として、オブジェクト側にミュータビリティという概念が必要になった、とも言える。